[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/session1/notebooks/S1_NB01_premier_contact.ipynb)

# S1-NB01 — Premier contact : la machine écoute le fongbe et le wolof

**Module 1 — Session 1 : Panorama de l'IA vocale & phonétique des langues locales** · Atelier 1A · 30 min · Ticket `M1-S1-04`

## 🎯 Objectifs
- Transcrire de la parole (ASR) et synthétiser de la parole (TTS) avec des modèles pré-entraînés, en quelques lignes.
- Constater qu'un modèle **généraliste** célèbre (Whisper) échoue sur le fongbe et le wolof — et comprendre **pourquoi**.
- Comparer avec un modèle **massivement multilingue** (MMS) et un modèle **spécialisé Afrique** (Simba, 2025).
- Développer un regard critique : « ça a l'air de marcher » ≠ « ça marche ».

## 🧭 Comment travailler
| Badge | Pour qui | Attendu |
|---|---|---|
| 🟢 Essentiel | Tout le monde | Exécuter, écouter, observer, répondre aux questions |
| 🔵 Intermédiaire | À l'aise en Python | Compléter le code marqué `✍️ À COMPLÉTER` |
| 🔴 Avancé | Experts ML | Défis ouverts (optionnels) |

Binôme **pilote / navigateur** : le pilote exécute, le navigateur lit les consignes, écoute et note les observations.
Le parcours 🟢 se fait **sans écrire de code** (≈ 20 min).

## 0. Préchauffage — à lancer dès l'ouverture de la session

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
import pandas as pd
from IPython.display import display

from m1utils.asr import load_asr, load_tts, transcribe_batch, unload, whisper_detect_language
from m1utils.audio import load_audio, play
from m1utils.data import load_cached_predictions, load_manifest
from m1utils.metrics import cer

demo = load_manifest("demo_s1")  # 4 fon + 6 wol + 2 fra, avec transcriptions de référence
display(demo[["id", "lang", "text", "duration_s", "source"]])

## 1. Écouter 🟢
Avant toute machine, **écoutons**. Le navigateur lit les transcriptions de référence pendant que vous écoutez.

Remarquez : les énoncés *lus* (ALFFA) et *spontanés* (Kallaama, radio) ; le français au **téléphone** (MInDS-14).

In [ ]:
for _, ligne in demo.iterrows():
    play(ligne.abspath, label=f"[{ligne.lang}] {ligne.text}")

## 2. Whisper sur du français 🟢
**Whisper** (OpenAI, 2022) a été entraîné sur 680 000 heures d'audio du web dans ≈ 100 langues.
Commençons par ce qu'il sait bien faire.

In [ ]:
whisper = load_asr("whisper-small")
francais = demo[demo.lang == "fra"]
for _, ligne in francais.iterrows():
    sortie = whisper(ligne.abspath)
    print(f"Référence : {ligne.text}\nWhisper   : {sortie.text}\n")

**❓ Question :** la transcription est-elle utilisable ? Comparez **mot à mot** avec la référence : qui a raison ?

✍️ *Votre réponse :* …

## 3. Whisper sur du fongbe et du wolof 🟢
Même modèle, mêmes réglages. Le navigateur note ce qui se passe.

In [ ]:
locales = demo[demo.lang != "fra"]
for _, ligne in locales.iterrows():
    sortie = whisper(ligne.abspath)
    print(f"[{ligne.lang}] Référence : {ligne.text}\n      Whisper   : {sortie.text[:150]}\n")

### Dans quelle langue Whisper croit-il être ? 🟢
Avant de transcrire, Whisper **devine la langue** parmi celles qu'il connaît.

In [ ]:
for _, ligne in demo.iterrows():
    hypotheses = whisper_detect_language(whisper, ligne.abspath, top_k=3)
    print(f"[{ligne.lang}] Whisper pense : " + ", ".join(f"{code} ({p:.0%})" for code, p in hypotheses))

**❓ Questions :**
1. Quelles langues Whisper « entend »-il pour le fongbe ? pour le wolof ? Pourquoi celles-là ?
2. Que produit-il alors comme texte ? Pourquoi certaines sorties sont-elles dans un autre **alphabet** ?
3. Pourquoi un modèle entraîné sur 680 000 heures échoue-t-il ici ?

✍️ *Votre réponse :* …

In [ ]:
unload(whisper)  # libérer la mémoire GPU avant le modèle suivant

## 4. MMS : un modèle pour plus de 1 000 langues 🟢
**MMS** (Meta, 2023) couvre 1 162 langues grâce à des enregistrements de textes religieux lus.
Il utilise un petit **adaptateur** par langue : on lui dit quelle langue il écoute.

In [ ]:
resultats = []
for langue in ["fon", "wol"]:
    mms = load_asr("mms-1b-all", lang=langue)
    sous_ensemble = locales[locales.lang == langue]
    for _, ligne in sous_ensemble.iterrows():
        sortie = mms(ligne.abspath)
        resultats.append({"id": ligne.id, "lang": langue, "référence": ligne.text, "MMS": sortie.text})
        print(f"[{langue}] Référence : {ligne.text}\n      MMS       : {sortie.text}\n")
    unload(mms)

**❓ Question :** comparez à Whisper. Que remarquez-vous dans les sorties fongbe de MMS (regardez les accents) ?

✍️ *Votre réponse :* …

### Exercice 4.1 🔵 — Mesurer l'erreur
Le **taux d'erreur sur les caractères** (*Character Error Rate*, CER) mesure la proportion de caractères à corriger
pour passer de la sortie du modèle à la référence (0 = parfait ; détails en session 3).
Calculez le CER de MMS sur chaque énoncé avec `cer(reference, hypothese)` et ajoutez-le au tableau.

In [ ]:
tableau = pd.DataFrame(resultats)
tableau["CER MMS"] = ...  # ✍️ À COMPLÉTER : une valeur par ligne
display(tableau.round(2))
assert "CER MMS" in tableau and tableau["CER MMS"].between(0, 5).all(), "Une valeur de CER par ligne est attendue."

## 5. Simba : l'état de l'art spécialisé Afrique (2025) 🟢
**Simba** (UBC, EMNLP 2025) est une famille de modèles fine-tunés sur SimbaBench, un ensemble de corpus de
43 langues africaines dont le fongbe et le wolof. **Simba-H** ne fait que 94 millions de paramètres (10 fois moins que MMS).

In [ ]:
simba = load_asr("simba-h")
tableau["Simba-H"] = [simba(chemin).text for chemin in locales.set_index("id").loc[tableau.id, "abspath"]]
unload(simba)
display(tableau[["lang", "référence", "MMS", "Simba-H"]])

### Exercice 5.1 🔵 — Qui gagne ?
Ajoutez la colonne `CER Simba-H`, puis affichez le **CER moyen par langue** pour les deux modèles
(`groupby("lang")`).

In [ ]:
tableau["CER Simba-H"] = ...  # ✍️ À COMPLÉTER
moyennes = ...                # ✍️ À COMPLÉTER : CER moyen par langue, deux colonnes
display(moyennes.round(2))

**❓ Question :** les énoncés wolof **spontanés** (Kallaama) sont-ils mieux ou moins bien reconnus que les énoncés lus ?
Pourquoi ?

✍️ *Votre réponse :* …

## 6. Et si on se trompe de langue ? 🔵
Chargez MMS avec l'adaptateur **wolof** et faites-lui écouter du **fongbe**. Que se passe-t-il ?

In [ ]:
fon_audio = locales[locales.lang == "fon"].abspath.iloc[0]
mms_wol = ...  # ✍️ À COMPLÉTER : MMS avec l'adaptateur "wol"
print("Référence (fon)      :", locales[locales.lang == "fon"].text.iloc[0])
print("MMS adaptateur wolof :", mms_wol(fon_audio).text)
unload(mms_wol)

✍️ *Votre réponse :* …

## 7. La synthèse vocale (TTS) 🟢
Le chemin inverse : du texte vers la voix. Deux architectures différentes :
- **fongbe** : MMS-TTS, un modèle **VITS** qui produit directement l'onde sonore ;
- **wolof** : **SpeechT5** produit un spectrogramme, puis un **vocodeur** (HiFi-GAN) le transforme en son.

In [ ]:
tts_fon = load_tts("mms-tts-fon")
phrase_fon = "a xa akwɛ ɔ a"
signal, sr = tts_fon(phrase_fon)
play(signal, sr, label=f"🗣️ fongbe (synthèse) : « {phrase_fon} »")

In [ ]:
tts_wol = load_tts("speecht5-wol")
phrase_wol = "na nga def, jàmm ngaam ?"
signal, sr = tts_wol(phrase_wol)
play(signal, sr, label=f"🗣️ wolof (synthèse) : « {phrase_wol} »")

### Exercice 7.1 🔵 — Les tons comptent-ils pour la machine ?
Le vocabulaire de MMS-TTS fongbe contient des lettres à tons (á, à, ǎ…). Synthétisez **la même phrase avec et sans
tons** et écoutez la différence. Exemple (tons à faire valider par un locuteur) : `"a ni kpɔ́n nǔ ɖé"` / `"a ni kpɔn nu ɖe"`.

In [ ]:
avec_tons, sans_tons = "...", "..."  # ✍️ À COMPLÉTER
for texte in (avec_tons, sans_tons):
    signal, sr = tts_fon(texte)
    play(signal, sr, label=f"« {texte} »")

**❓ Question :** entendez-vous une différence de mélodie ? Qu'est-ce que cela implique pour les textes qu'on donne à un
système de synthèse, sachant que la plupart des textes fongbe du web n'ont **pas** de tons ?

✍️ *Votre réponse :* …

## 8. À vous de parler ! 🟢 (optionnel, Colab + micro)
Passez `ENREGISTRER = True`, autorisez le micro et dites une phrase dans votre langue (fongbe ou wolof).

In [ ]:
ENREGISTRER = False
LANGUE = "fon"  # "fon" ou "wol"

if ENREGISTRER:
    from m1utils.audio import record
    y, sr = record(seconds=5)
    play(y, sr, label="Votre voix :")
    modele = load_asr("simba-h")
    print("Simba-H a compris :", modele(y).text)
    unload(modele)

## 9. Défi 🔴 — La boucle TTS → ASR
Synthétisez 3 phrases wolof avec SpeechT5, transcrivez-les avec Simba-H, et calculez le CER.
Que mesure ce chiffre ? Que **ne** mesure-t-il **pas** ?

In [ ]:
...  # ✍️ À COMPLÉTER

✍️ *Votre réponse :* …

In [ ]:
unload(tts_fon, tts_wol)

## ✅ Bilan — à compléter en binôme (sert pour la restitution et le TD1)

| | Ce qui marche | Ce qui ne marche pas |
|---|---|---|
| Whisper | ✍️ | ✍️ |
| MMS | ✍️ | ✍️ |
| Simba-H | ✍️ | ✍️ |
| TTS | ✍️ | ✍️ |

**À retenir**
- Un modèle généraliste, même énorme, **ne couvre pas** les langues absentes de ses données : il hallucine.
- Les modèles **multilingues** (MMS) et **spécialisés** (Simba) donnent des résultats exploitables mais imparfaits,
  surtout sur la parole **spontanée**.
- L'**écriture** (tons, caractères ɛ/ε) influence à la fois la mesure de l'erreur et la synthèse.

**Prochaine étape :** atelier 1B — voir les tons et les voyelles dans le signal, et maîtriser l'écriture numérique.

🛟 *Plan B* (si un modèle ne se charge pas) : `load_cached_predictions("simba-h", "fon")` donne des sorties pré-calculées.